# S51 후속 답변 방식 고르기

개발용 대화 10개(후속 답변 20개)의 실행 기록으로 앞 대화 형식과 앞 근거 판단 방식을 고른다. 고르는 기준은 실행 전에 정했다(`docs/slices/51-evidence-reuse.md` 결정 사항 3). 실행 기록은 `rag.multiturn_check`가 만들었고 이 노트북은 API를 부르지 않는다.

In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / "data/multiturn/runs"
FORMATS = ["turns", "block"]
MODES = ["off", "rule", "rewrite", "decision"]

In [3]:
def load(name: str) -> list[dict]:
    with (RUNS / f"dev-{name}.jsonl").open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def follow_ups(rows: list[dict]) -> pd.DataFrame:
    """후속 답변(2번째 턴부터)의 검사 값과 라벨."""
    fu = [r for r in rows if r["turn"] > 1]
    return pd.DataFrame({
        "cid": [r["cid"] for r in fu],
        "turn": [r["turn"] for r in fu],
        "type": [r["label"]["type"] for r in fu],
        "needs_prev": [r["label"]["needs_prev"] for r in fu],
        "reused": [bool(r["reused_docs"]) for r in fu],
        "uncited": [r["checks"]["uncited"] for r in fu],
        "repeat": [r["checks"]["repeat"] for r in fu],
        "repeat_ratio": [r["checks"]["repeat_ratio"] for r in fu],
        "refusal": [r["checks"]["refusal"] for r in fu],
        "rewrite_s": [r["meta"]["rewrite_s"] for r in fu],
        "decision_s": [r["meta"].get("decision_s") for r in fu],
        "first_token_s": [r["first_token_s"] for r in fu],
    })

## 앞 대화 형식: turns와 block

앞 근거 다시 쓰기는 끈 상태(`off`)로 비교한다.

In [4]:
fmt = {f: follow_ups(load(f"{f}-off")) for f in FORMATS}
pd.DataFrame({f: {"후속 답변": len(d), "인용 누락": int(d.uncited.sum()), "거부 없는 되풀이": int(d.repeat.sum()), "거부 답변": int(d.refusal.notna().sum())} for f, d in fmt.items()}).T

,후속 답변,인용 누락,거부 없는 되풀이,거부 답변
turns,20,1,0,8
block,20,0,1,8


기계 검사에 걸린 답변을 읽는다. 대화 질문은 메인이 지은 것이고 답변은 공개 회의록 근거로 만든 글이다.

In [5]:
for f in FORMATS:
    rows = {(r["cid"], r["turn"]): r for r in load(f"{f}-off")}
    d = fmt[f]
    for _, x in d[d.uncited | d.repeat].iterrows():
        r, prev = rows[(x.cid, x.turn)], rows[(x.cid, x.turn - 1)]
        print(f"[{f}] {x.cid} {x.turn}번째 턴 ({x.type}), 인용 누락 {x.uncited}, 되풀이 {x.repeat} ({x.repeat_ratio})")
        print("  앞 답변:", prev["raw"][:300])
        print("  질문:", r["q"])
        print("  답변:", r["raw"][:300])
        print()

[turns] c15 3번째 턴 (switch), 인용 누락 True, 되풀이 <bound method Series.repeat of cid                   c15
turn                    3
type               switch
needs_prev          False
reused              False
uncited              True
repeat              False
repeat_ratio        0.197
refusal               NaN
rewrite_s           0.868
decision_s           None
first_token_s    0.686184
Name: 17, dtype: object> (0.197)
  앞 답변: 제공된 회의록에서 답을 찾지 못했습니다.
  질문: 국회의장 했던 사람이 총리 맡는 게 입법부 권위 문제라는 얘기 청문회에서 나왔어?
  답변: 2020년 1월 7일 국무총리(정세균)임명동의에관한인사청문특별위원회 제374회 제3차 회의에서 윤소하 위원은 일각에서 국회의장을 역임한 후보자가 국무총리를 맡은 것에 대해 입법부 권위 훼손이 아니냐는 이야기가 있다는 발언을 했습니다.

[block] c12 2번째 턴 (refer), 인용 누락 False, 되풀이 <bound method Series.repeat of cid                  c12
turn                   2
type               refer
needs_prev          True
reused             False
uncited            False
repeat              True
repeat_ratio       0.756
refusal              NaN
rewrite_s          6.617
decision_s          None
first_tok

`turns`의 c15 3번째 턴은 주제를 바꾼 질문에 인용 없이 답했다(실제 위반). `block`의 c12 2번째 턴은 첫 답변이 이미 담은 정부 답을 다시 묻는 질문에 맞게 답했다(위반 아님). 기준을 글자 그대로 적용하면 동점이라 `turns`지만, 확인한 위반만 세는 메인 권고에 따라 사용자가 `block`을 골랐다(2026-11-06).

## 앞 근거 판단: off, rule, rewrite, decision

형식은 `block`이다. 라벨 `needs_prev`는 표본을 지을 때 후속 질문마다 적은 "앞 근거가 다시 필요한가"다. `reused`는 앞 근거를 실제로 근거 칸에 넣었는지다.

In [6]:
mode = {m: follow_ups(load(f"block-{m}")) for m in MODES}
summary = {}
for m, d in mode.items():
    summary[m] = {
        "라벨 일치": int((d.needs_prev == d.reused).sum()),
        "필요해서 넣음": int((d.needs_prev & d.reused).sum()),
        "불필요한데 넣음": int((~d.needs_prev & d.reused).sum()),
        "필요한데 안 넣음": int((d.needs_prev & ~d.reused).sum()),
        "인용 누락": int(d.uncited.sum()),
        "거부 없는 되풀이": int(d.repeat.sum()),
        "재작성 중앙값(초)": round(d.rewrite_s.median(), 2),
        "판단 중앙값(초)": round(d.decision_s.dropna().median(), 2) if d.decision_s.notna().any() else None,
        "생성 첫 토큰 중앙값(초)": round(d.first_token_s.median(), 2),
    }
pd.DataFrame(summary).T

,라벨 일치,필요해서 넣음,불필요한데 넣음,필요한데 안 넣음,인용 누락,거부 없는 되풀이,재작성 중앙값(초),판단 중앙값(초),생성 첫 토큰 중앙값(초)
off,8.0,0.0,0.0,12.0,0.0,1.0,1.21,NaN,0.77
rule,18.0,11.0,1.0,1.0,0.0,1.0,0.66,NaN,0.77
rewrite,16.0,8.0,0.0,4.0,0.0,1.0,0.74,NaN,0.77
decision,10.0,5.0,3.0,7.0,1.0,1.0,0.69,0.67,0.74


라벨과 어긋난 후속 질문(방식별).

In [7]:
for m in ["rule", "rewrite", "decision"]:
    d = mode[m]
    miss = d[d.needs_prev != d.reused]
    print(m, miss[["cid", "turn", "type", "needs_prev", "reused"]].to_string(index=False))
    print()

rule cid  turn  type  needs_prev  reused
c01     3 widen       False    True
c06     3 widen        True   False

rewrite cid  turn   type  needs_prev  reused
c06     3  widen        True   False
c09     2  refer        True   False
c11     2 detail        True   False
c15     2 detail        True   False

decision cid  turn   type  needs_prev  reused
c01     3  widen       False    True
c03     2 detail        True   False
c03     3  widen       False    True
c06     3  widen        True   False
c09     2  refer        True   False
c09     3 detail        True   False
c11     2 detail        True   False
c13     2 detail        True   False
c15     2 detail        True   False
c16     2  widen       False    True



## 고른 방식

- 형식: `block`(위 형식 절, 사용자 결정 2026-11-06)
- 판단: `rule`. 기준은 라벨 일치가 가장 많은 쪽이고, 2개 차이 안이면 호출이 적은 쪽(`rule` > `rewrite` > `decision`)이다. `rule`이 가장 많고 호출도 없다. 판단 모델(`decision`)은 앞 근거를 넣지 않는 `off`와 거의 같았다.
- 한계: 라벨은 subagent가 지었고 애매한 칸이 있다(widen 유형 등). 로컬은 재정렬을 꺼 `rule`이 보는 후보가 Space와 다르다. 답변이 나아졌는지는 확인용 10개를 사용자가 읽고 판단한다.

## 확인용 뒤 수정: 앞 근거 중복을 청크 단위로

확인용 c10 2번째 턴에서 앞 근거가 같은 회의록의 새 청크(질문한 대목)를 밀어내 답이 다른 회의로 갔다. 문서 단위로 중복을 걸렀기 때문이라 청크 단위로 고쳤다(b509cfc, 사용자 승인 2026-11-06). 확인용은 다시 돌리지 않고 개발용 `rule`을 다시 돌렸다(`dev-block-rule-chunk`).

In [8]:
fixed = follow_ups(load("block-rule-chunk"))
before = mode["rule"]
pd.DataFrame({
    "고치기 전": {"라벨 일치": int((before.needs_prev == before.reused).sum()), "인용 누락": int(before.uncited.sum()), "거부 없는 되풀이": int(before.repeat.sum()), "전체 거부": int((before.refusal == "full").sum())},
    "고친 뒤": {"라벨 일치": int((fixed.needs_prev == fixed.reused).sum()), "인용 누락": int(fixed.uncited.sum()), "거부 없는 되풀이": int(fixed.repeat.sum()), "전체 거부": int((fixed.refusal == "full").sum())},
})

,고치기 전,고친 뒤
라벨 일치,18,18
인용 누락,0,0
거부 없는 되풀이,1,1
전체 거부,7,5


## block_v2: 더 파고드는 질문의 되풀이

Space 미리보기에서 사용자 대화의 2번째 턴("…자세히 알려줘")이 여전히 앞 답을 되풀이했다. `block_v2`는 근거에 더한 내용이 없으면 찾지 못했다고 쓰라는 줄을 더했다(사용자 승인 2026-11-06). 채택 기준은 사용자 대화 2번째 턴이 되풀이하지 않고, 개발용이 나빠지지 않는 것이다. 로컬은 재정렬을 꺼 그 대화의 첫 답과 근거가 Space와 달라 되풀이가 재현되지 않았다(v1과 v2 모두 새 내용으로 답함). 개발용 비교:

In [9]:
v2 = follow_ups(load("block-rule-chunk-b2"))
cols = {"라벨 일치": lambda d: int((d.needs_prev == d.reused).sum()), "인용 누락": lambda d: int(d.uncited.sum()), "거부 없는 되풀이": lambda d: int(d.repeat.sum()), "전체 거부": lambda d: int((d.refusal == "full").sum())}
print(pd.DataFrame({"block_v1": {k: f(fixed) for k, f in cols.items()}, "block_v2": {k: f(v2) for k, f in cols.items()}}))
changed = fixed.refusal.fillna("-") != v2.refusal.fillna("-")
print(pd.DataFrame({"cid": fixed.cid, "turn": fixed.turn, "type": fixed.type, "v1": fixed.refusal, "v2": v2.refusal})[changed].to_string(index=False))

           block_v1  block_v2
라벨 일치            18        17
인용 누락             0         0
거부 없는 되풀이         1         1
전체 거부             5         6
cid  turn  type  v1   v2
c06     2 refer NaN full


c06 2번째 턴(지시어 풀기, 답이 있는 질문)을 v1은 답했고 v2는 전부 거부했다. 개발용이 나빠져 기준에 걸리므로 v1을 유지한다. 사용자 대화의 되풀이는 S51 관찰로 남긴다.